In [0]:
# 1. cargar librerias iniciales
from pyspark.sql.functions import *

In [0]:
# 2. Crear la tabla workspace.weather_gold.fact_weather
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.weather_gold.fact_weather (
  id STRING,
  date_id INTEGER,
  location_id STRING,
  temp_max DOUBLE,
  temp_min DOUBLE,
  precipitacion DOUBLE,
  velocidad_viento DOUBLE
)
""")

In [0]:
# 2. Leer bronze y pasar a pandas
df_silver = spark.table("workspace.weather_silver.weather")
df_silver=df_silver.toPandas()

# 3. Crear FACT_WEATHER
fact_weather = (
    df_silver[
        [
            "id",
            "date_id",
            "location_id",
            "daily_temperature_2m_max",
            "daily_temperature_2m_min",
            "daily_precipitation_sum",
            "daily_wind_speed_10m_max"
        ]
    ]
    .rename(
        columns={
            "daily_temperature_2m_max": "temp_max",
            "daily_temperature_2m_min": "temp_min",
            "daily_precipitation_sum": "precipitacion",
            "daily_wind_speed_10m_max": "velocidad_viento"
        }
    )
)

# 3. Verificar
print(fact_weather.head())
print(fact_weather.dtypes)

# 4. Pandas -> Spark
fact_weather_spark = spark.createDataFrame(
    fact_weather
)

fact_weather_spark.printSchema()

# 5. Guardar en GOLD (overwriteSchema permite actualizar el esquema)
fact_weather_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "workspace.weather_gold.fact_weather"
    )

In [0]:
%sql
select * from workspace.weather_gold.fact_weather limit 5